# ShopFlow — Phase 6 Visual Analytics
### Python-powered charts on the ANALYTICS layer — built inside Snowflake Notebooks

---

**What you'll build in this phase:**
- Monthly GMV trend (line chart)
- Review score vs delivery delay (scatter + trend line)
- Top 10 product categories by revenue (horizontal bar)
- Seller performance bubble chart (orders × GMV × review score)
- Orders by Brazilian state (choropleth map)

**Prerequisites:** Phase 03 complete — `SHOPFLOW_ANALYTICS` schema must exist with `FACT_ORDERS`, all DIM tables, and `DAILY_SELLER_SUMMARY`

**Tools:** Snowpark (Python) · Plotly · Pandas · all pre-installed in Snowflake Notebooks — no pip install needed

---

> **💡 How Python + SQL work together in Snowflake Notebooks**
>
> Each cell can be SQL or Python — switch via the language selector top-right of each cell.
>
> In Python cells, use `session.sql("...")` to run SQL and get back a **Snowpark DataFrame**.
> Call `.to_pandas()` to convert it for plotting with Plotly or Matplotlib.
>
> ```python
> df = session.sql("SELECT * FROM FACT_ORDERS LIMIT 10").to_pandas()
> ```
>
> The `session` object is **automatically available** in every Python cell — you do not need to create a Snowflake connection. Snowflake injects it for you.

---
## Cell 1 — Set context and verify ANALYTICS layer

Before plotting anything, confirm you are pointing at the right database and schema, and that all required tables exist with the expected row counts.

In [ ]:
-- Set context
--USE WAREHOUSE SHOPFLOW_WH;   -- uncomment if not set automatically
USE DATABASE SHOPFLOW_DB;
USE SCHEMA   SHOPFLOW_ANALYTICS;

In [ ]:
-- Verify all ANALYTICS tables exist with data
SELECT
    table_name,
    row_count,
    ROUND(bytes / 1024 / 1024, 2) AS size_mb
FROM SHOPFLOW_DB.INFORMATION_SCHEMA.TABLES
WHERE table_schema = 'SHOPFLOW_ANALYTICS'
  AND table_type  = 'BASE TABLE'
ORDER BY table_name;

In [ ]:
-- Confirm session context before running any Python cells
-- All session info via SQL — no Python session.get_* needed
SELECT
    CURRENT_WAREHOUSE() AS warehouse,
    CURRENT_DATABASE()  AS database,
    CURRENT_SCHEMA()    AS schema,
    CURRENT_ROLE()      AS role;

---
## Cell 2 — Import libraries

All libraries are pre-installed in Snowflake Notebooks. No `pip install` required.

> **💡 Pro Tip — Snowpark vs Pandas**
>
> `snowflake.snowpark` lets you write DataFrame transformations that run **inside Snowflake** as SQL under the hood — no data leaves the platform until you call `.to_pandas()`.
>
> For small result sets (< 100k rows) used for plotting, `.to_pandas()` is fine. For large transforms, keep the logic in Snowpark or SQL to avoid pulling millions of rows into the notebook runtime.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import streamlit as st

print('✅ Session and libraries ready')
print(f'   Pandas : {pd.__version__}')
print(f'   DB     : {session.get_current_database()}')
print(f'   Schema : {session.get_current_schema()}')

---
## Cell 3 — Chart 1 · Monthly GMV trend

**Business question:** How did ShopFlow's total revenue grow month by month?

GMV = Gross Merchandise Value = sum of `price + freight_value` for all delivered orders.

> **💡 SnowPro Tip — DATE_TRUNC**
>
> `DATE_TRUNC('month', ts)` truncates a TIMESTAMP to the first day of its month — perfect for time-series grouping. Works on TIMESTAMP_NTZ, DATE, and TIMESTAMP_LTZ.
>
> The `YEAR(ts)` and `MONTH(ts)` functions also extract parts, but DATE_TRUNC returns a sortable date value you can pass directly to plotting libraries.

In [ ]:
-- Monthly GMV — only delivered orders, ordered chronologically
SELECT
    DATE_TRUNC('month', order_purchase_ts)      AS order_month,
    COUNT(DISTINCT order_id)                    AS orders,
    ROUND(SUM(price + freight_value), 2)        AS gmv_brl,
    ROUND(AVG(price + freight_value), 2)        AS avg_order_value_brl
FROM SHOPFLOW_ANALYTICS.FACT_ORDERS
WHERE order_status = 'delivered'
  AND order_purchase_ts >= '2017-01-01'
  AND order_purchase_ts <  '2018-10-01'   -- exclude sparse months at edges
GROUP BY 1
ORDER BY 1;

In [ ]:
# Pull SQL result into Pandas
df_gmv = session.sql("""
    SELECT
        DATE_TRUNC('month', order_purchase_ts)   AS order_month,
        COUNT(DISTINCT order_id)                 AS orders,
        ROUND(SUM(price + freight_value), 2)     AS gmv_brl,
        ROUND(AVG(price + freight_value), 2)     AS avg_order_value_brl
    FROM SHOPFLOW_ANALYTICS.FACT_ORDERS
    WHERE order_status = 'delivered'
      AND order_purchase_ts >= '2017-01-01'
      AND order_purchase_ts <  '2018-10-01'
    GROUP BY 1
    ORDER BY 1
""").to_pandas()

df_gmv['ORDER_MONTH'] = pd.to_datetime(df_gmv['ORDER_MONTH'])

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Bar(
        x=df_gmv['ORDER_MONTH'],
        y=df_gmv['GMV_BRL'],
        name='GMV (BRL)',
        marker_color='#29B5E8',
        opacity=0.85
    ),
    secondary_y=False
)

fig.add_trace(
    go.Scatter(
        x=df_gmv['ORDER_MONTH'],
        y=df_gmv['ORDERS'],
        name='Orders',
        mode='lines+markers',
        line=dict(color='#FF6B6B', width=2.5),
        marker=dict(size=7)
    ),
    secondary_y=True
)

fig.update_layout(
    title=dict(text='Monthly GMV & Order Count — ShopFlow Analytics', font=dict(size=18)),
    xaxis_title='Month',
    plot_bgcolor='#f8fafc',
    paper_bgcolor='white',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    hovermode='x unified'
)
fig.update_yaxes(title_text='GMV (R$)', secondary_y=False)
fig.update_yaxes(title_text='Order Count', secondary_y=True)

st.plotly_chart(fig, use_container_width=True)
print(f"Peak GMV month: {df_gmv.loc[df_gmv['GMV_BRL'].idxmax(), 'ORDER_MONTH'].strftime('%B %Y')} — R$ {df_gmv['GMV_BRL'].max():,.0f}")

---
## Cell 4 · Chart 2 — Review score vs delivery delay

**Business question:** Does delivering late actually hurt review scores? By how many days before customers start complaining?

This scatter plot bins orders by delivery delay (days early/late) and shows the average review score per bucket. A clear inflection point should appear around 0–5 days late.

> **💡 SnowPro Tip — DATEDIFF**
>
> `DATEDIFF('day', estimated_delivery_ts, delivered_ts)` returns positive values for late deliveries and negative for early ones. Snowflake's `DATEDIFF` takes the unit first (unlike some other databases where column order is reversed — a common exam trap).

In [ ]:
import numpy as np

df_reviews = session.sql("""
    SELECT
        DATEDIFF('day', estimated_delivery_ts, delivered_ts) AS delay_days,
        COUNT(*)                                              AS order_count,
        ROUND(AVG(r.review_score), 3)                        AS avg_review_score
    FROM SHOPFLOW_ANALYTICS.FACT_ORDERS f
    JOIN SHOPFLOW_DB.SHOPFLOW_STAGED.STG_REVIEWS r
      ON f.order_id = r.order_id
    WHERE f.delivered_ts IS NOT NULL
      AND r.review_score IS NOT NULL
      AND DATEDIFF('day', estimated_delivery_ts, delivered_ts) BETWEEN -30 AND 30
    GROUP BY 1
    HAVING COUNT(*) >= 20
    ORDER BY 1
""").to_pandas()

# Fit a simple trend line with numpy (no statsmodels needed)
x = df_reviews['DELAY_DAYS'].values
y = df_reviews['AVG_REVIEW_SCORE'].values
z = np.polyfit(x, y, 2)          # degree-2 polynomial
p = np.poly1d(z)
x_line = np.linspace(x.min(), x.max(), 100)
y_line = p(x_line)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_reviews['DELAY_DAYS'],
    y=df_reviews['AVG_REVIEW_SCORE'],
    mode='markers',
    marker=dict(
        size=df_reviews['ORDER_COUNT'] / df_reviews['ORDER_COUNT'].max() * 30 + 5,
        color=df_reviews['AVG_REVIEW_SCORE'],
        colorscale=['#FF4444', '#FFA500', '#29B5E8'],
        showscale=False
    ),
    name='Delay bucket',
    hovertemplate='Delay: %{x}d<br>Score: %{y:.2f}<extra></extra>'
))
fig.add_trace(go.Scatter(
    x=x_line, y=y_line,
    mode='lines',
    line=dict(color='#0d1f3c', width=2, dash='dot'),
    name='Trend'
))
fig.add_vline(x=0, line_dash='dash', line_color='#888',
              annotation_text='On time', annotation_position='top')
fig.update_layout(
    title='Review Score vs Delivery Delay — does late delivery hurt ratings?',
    xaxis_title='Delivery Delay (days · negative = early)',
    yaxis_title='Avg Review Score (1–5)',
    plot_bgcolor='#f8fafc',
    paper_bgcolor='white'
)
st.plotly_chart(fig, use_container_width=True)

on_time = df_reviews[df_reviews['DELAY_DAYS'] <= 0]['AVG_REVIEW_SCORE'].mean()
late    = df_reviews[df_reviews['DELAY_DAYS'] >  0]['AVG_REVIEW_SCORE'].mean()
print(f'Avg score — on time or early : {on_time:.2f}')
print(f'Avg score — late             : {late:.2f}')
print(f'Score drop for late delivery : {on_time - late:.2f} points')

---
## Cell 5 · Chart 3 — Top 10 product categories by revenue

**Business question:** Which product categories drive the most GMV on the Olist platform?

Uses `DIM_PRODUCTS` which already has English category names joined from `RAW_CATEGORY_TRANSLATION` in Phase 03.

> **💡 SnowPro Tip — NULLIF and COALESCE**
>
> Some products have no category name in the source data. `COALESCE(category_english, 'uncategorised')` handles those gracefully — returning the fallback string instead of NULL. NULLs in GROUP BY are treated as a single group in Snowflake, so without COALESCE you would see a NULL row in your results.

In [ ]:
df_cat = session.sql("""
    SELECT
        COALESCE(p.category_english, 'uncategorised') AS category,
        ROUND(SUM(f.price + f.freight_value), 0)      AS gmv_brl,
        COUNT(*)                                       AS items_sold,
        ROUND(AVG(f.price), 2)                         AS avg_price_brl
    FROM SHOPFLOW_ANALYTICS.FACT_ORDERS f
    JOIN SHOPFLOW_ANALYTICS.DIM_PRODUCTS p
      ON f.product_id = p.product_id
    WHERE f.order_status = 'delivered'
    GROUP BY 1
    ORDER BY gmv_brl DESC
    LIMIT 10
""").to_pandas()

df_cat = df_cat.sort_values('GMV_BRL')  # ascending for horizontal bar

fig = px.bar(
    df_cat,
    x='GMV_BRL',
    y='CATEGORY',
    orientation='h',
    color='GMV_BRL',
    color_continuous_scale=['#B3E5FC', '#29B5E8', '#0d1f3c'],
    text=df_cat['GMV_BRL'].apply(lambda x: f'R$ {x/1e6:.1f}M'),
    labels={'GMV_BRL': 'GMV (BRL)', 'CATEGORY': 'Product Category'},
    title='Top 10 Product Categories by GMV — delivered orders'
)

fig.update_traces(textposition='outside')
fig.update_layout(
    plot_bgcolor='#f8fafc',
    paper_bgcolor='white',
    coloraxis_showscale=False,
    xaxis_tickformat=',.0f',
    margin=dict(l=200)
)
st.plotly_chart(fig, use_container_width=True)
print(f"Top category: {df_cat.iloc[-1]['CATEGORY']} — R$ {df_cat.iloc[-1]['GMV_BRL']:,.0f}")

---
## Cell 6 · Chart 4 — Seller performance bubble chart

**Business question:** Which sellers are high-volume + high-rated? Which are high-volume but low-rated (churn risk)?

Each bubble = one seller. X axis = order count, Y axis = average review score, bubble size = total GMV. Color encodes performance tier.

> **💡 SnowPro Tip — QUALIFY with aggregates**
>
> We filter to sellers with at least 30 orders using `HAVING COUNT(*) >= 30` in the subquery. Alternatively in Snowflake you could use `QUALIFY ROW_NUMBER() OVER (ORDER BY order_count DESC) <= 200` to cap results to the top 200 sellers by volume — QUALIFY makes this one-line clean without a subquery.

In [ ]:
df_sellers = session.sql("""
    SELECT
        f.seller_id,
        COUNT(*)                                AS order_count,
        ROUND(SUM(f.price + f.freight_value), 0) AS gmv_brl,
        ROUND(AVG(r.review_score), 2)            AS avg_review_score,
        ROUND(AVG(
            DATEDIFF('day', f.estimated_delivery_ts, f.delivered_ts)
        ), 1)                                    AS avg_delay_days
    FROM SHOPFLOW_ANALYTICS.FACT_ORDERS f
    LEFT JOIN SHOPFLOW_DB.SHOPFLOW_STAGED.STG_REVIEWS r
           ON f.order_id = r.order_id
    WHERE f.order_status = 'delivered'
      AND f.delivered_ts IS NOT NULL
    GROUP BY 1
    HAVING COUNT(*) >= 30
    ORDER BY gmv_brl DESC
""").to_pandas()

# Performance tier
def tier(row):
    score = row['AVG_REVIEW_SCORE']
    if pd.isna(score):                                    # no reviews yet
        return '📦 Standard'
    if score >= 4.2 and row['ORDER_COUNT'] >= 100:
        return '⭐ Star seller'
    elif score < 3.5:
        return '⚠️  At risk'
    else:
        return '📦 Standard'

df_sellers['tier'] = df_sellers.apply(tier, axis=1)

fig = px.scatter(
    df_sellers,
    x='ORDER_COUNT',
    y='AVG_REVIEW_SCORE',
    size='GMV_BRL',
    color='tier',
    hover_data=['SELLER_ID', 'GMV_BRL', 'AVG_DELAY_DAYS'],
    color_discrete_map={
        '⭐ Star seller': '#29B5E8',
        '📦 Standard':    '#A8D8EA',
        '⚠️  At risk':    '#FF6B6B'
    },
    labels={
        'ORDER_COUNT':       'Total Orders',
        'AVG_REVIEW_SCORE':  'Avg Review Score (1–5)',
        'GMV_BRL':           'GMV (BRL)'
    },
    title='Seller Performance — volume × rating × GMV'
)

fig.add_hline(y=4.0, line_dash='dot', line_color='#29B5E8',
              annotation_text='Score threshold 4.0', annotation_position='right')
fig.update_layout(
    plot_bgcolor='#f8fafc',
    paper_bgcolor='white',
    yaxis=dict(range=[1, 5.2])
)
st.plotly_chart(fig, use_container_width=True)

stars = (df_sellers['tier'] == '⭐ Star seller').sum()
at_risk = (df_sellers['tier'] == '⚠️  At risk').sum()
print(f"Star sellers  : {stars}")
print(f"At-risk sellers: {at_risk}")
print(f"Total sellers analysed: {len(df_sellers)}")

---
## Cell 7 · Chart 5 — Orders by Brazilian state (choropleth map)

**Business question:** Which Brazilian states have the highest order volume and GMV? Where is Olist's customer base concentrated?

Uses `DIM_CUSTOMERS` which already has `state` from `STG_CUSTOMERS`. Plotly Express has Brazil state codes built in — pass `locations='state'` and `locationmode='geojson-id'`.

> **💡 SnowPro Tip — DIM table value**
>
> This query is a single clean JOIN: `FACT_ORDERS → DIM_CUSTOMERS → aggregate by state`. Without the star schema you would need to join through STG_ORDERS → STG_CUSTOMERS every time. The DIM table collapses that complexity — exactly why star schemas exist.

In [ ]:
# Note: external URL fetching (urllib) is blocked in Snowflake Notebooks
# Using a bar chart ranked by state — same insight, no network call needed
# For the choropleth map, run this notebook locally in Jupyter with internet access

df_states = session.sql("""
    SELECT
        c.state                                          AS state_code,
        COUNT(*)                                         AS orders,
        COUNT(DISTINCT f.customer_id)                    AS unique_customers,
        ROUND(SUM(f.price + f.freight_value), 0)         AS gmv_brl,
        ROUND(AVG(f.price + f.freight_value), 2)         AS avg_order_value
    FROM SHOPFLOW_ANALYTICS.FACT_ORDERS f
    JOIN SHOPFLOW_ANALYTICS.DIM_CUSTOMERS c
      ON f.customer_id = c.customer_id
    WHERE f.order_status = 'delivered'
    GROUP BY 1
    ORDER BY orders DESC
""").to_pandas()

fig = px.bar(
    df_states,
    x='STATE_CODE',
    y='ORDERS',
    color='GMV_BRL',
    color_continuous_scale=['#B3E5FC', '#29B5E8', '#0d1f3c'],
    text='ORDERS',
    hover_data=['UNIQUE_CUSTOMERS', 'GMV_BRL', 'AVG_ORDER_VALUE'],
    labels={
        'STATE_CODE': 'Brazilian State',
        'ORDERS':     'Order Count',
        'GMV_BRL':    'GMV (BRL)'
    },
    title='Orders by Brazilian State — all delivered orders'
)
fig.update_traces(textposition='outside')
fig.update_layout(
    plot_bgcolor='#f8fafc',
    paper_bgcolor='white',
    coloraxis_showscale=False,
    xaxis_tickangle=-45
)
st.plotly_chart(fig, use_container_width=True)

print('Top 5 states by order volume:')
print(df_states[['STATE_CODE','ORDERS','GMV_BRL']].head(5).to_string(index=False))

---
## Cell 8 · Chart 6 — Payment method mix

**Business question:** How do customers pay? What share of GMV comes from credit card instalment plans vs other methods?

Uses `STG_PAYMENTS` — each order can have multiple payment rows (split payments). We aggregate to order level first before computing shares.

> **💡 SnowPro Tip — Aggregating before joining**
>
> `STG_PAYMENTS` has one row per payment instalment, not per order. Always aggregate to the order level first (in a CTE or subquery), then join to `FACT_ORDERS`. Joining first and aggregating later inflates row counts and produces incorrect sums — a classic data engineering mistake.

In [ ]:
df_pay = session.sql("""
    WITH order_payments AS (
        SELECT
            order_id,
            payment_type,
            SUM(TRY_CAST(payment_value AS FLOAT)) AS payment_value
        FROM SHOPFLOW_DB.SHOPFLOW_STAGED.STG_PAYMENTS
        GROUP BY 1, 2
    )
    SELECT
        payment_type,
        COUNT(DISTINCT order_id)               AS orders,
        ROUND(SUM(payment_value), 0)           AS total_value_brl,
        ROUND(AVG(payment_value), 2)           AS avg_value_brl
    FROM order_payments
    GROUP BY 1
    ORDER BY total_value_brl DESC
""").to_pandas()

fig = make_subplots(
    rows=1, cols=2,
    specs=[[{'type': 'pie'}, {'type': 'bar'}]],
    subplot_titles=['Share of Orders', 'Total Value (BRL)']
)

colors = ['#29B5E8', '#0d1f3c', '#FF6B6B', '#FFA500', '#A8D8EA']

fig.add_trace(
    go.Pie(
        labels=df_pay['PAYMENT_TYPE'],
        values=df_pay['ORDERS'],
        marker_colors=colors,
        hole=0.4,
        textinfo='label+percent'
    ),
    row=1, col=1
)

fig.add_trace(
    go.Bar(
        x=df_pay['PAYMENT_TYPE'],
        y=df_pay['TOTAL_VALUE_BRL'],
        marker_color=colors,
        text=df_pay['TOTAL_VALUE_BRL'].apply(lambda x: f'R$ {x/1e6:.1f}M'),
        textposition='outside'
    ),
    row=1, col=2
)

fig.update_layout(
    title='Payment Method Mix — orders and GMV',
    plot_bgcolor='#f8fafc',
    paper_bgcolor='white',
    showlegend=False,
    height=450
)
st.plotly_chart(fig, use_container_width=True)

top = df_pay.iloc[0]
print(f"Dominant payment type: {top['PAYMENT_TYPE']}")
print(f"  {top['ORDERS']:,} orders — R$ {top['TOTAL_VALUE_BRL']:,.0f} total")

---
## Cell 9 — Phase 6 summary

You've built 6 production-quality charts directly inside Snowflake using Python + Snowpark + Plotly.

| Chart | Business question answered |
|---|---|
| Monthly GMV trend | How is revenue growing over time? |
| Review score vs delivery delay | Does late delivery hurt ratings? |
| Top categories by GMV | Which product types drive revenue? |
| Seller performance bubbles | Which sellers are star vs at-risk? |
| Orders by state (map) | Where is our customer base? |
| Payment method mix | How do customers prefer to pay? |

---

> **💡 SnowPro Tip — Python in Snowflake (exam topics)**
>
> Snowflake Notebooks support Python via a **Snowpark** session. Key exam concepts:
>
> - `session.sql("...").to_pandas()` — run SQL, get Pandas DataFrame
> - `session.table("FACT_ORDERS")` — load a full table as Snowpark DataFrame
> - Python cells run on **Snowflake-managed compute** — same warehouse as SQL cells
> - Supported libraries: pandas, numpy, scikit-learn, matplotlib, plotly, altair — all pre-installed
> - For ML workloads, Snowpark ML provides feature engineering + model training inside Snowflake

---

**Project complete — all 6 phases done.**

```
Phase 00  ✅  Snowflake setup
Phase 01  ✅  Load RAW data
Phase 02  ✅  Build STAGED layer
Phase 03  ✅  ANALYTICS star schema
Phase 04  ✅  Performance + Security
Phase 05  ✅  Streams, Tasks & Sharing
Phase 06  ✅  Visual Analytics (this notebook)
```
